# Local-temperature heat-wave comparison: does station-scale warmth explain Sierra melt events better than synoptic Z500/T700 ridging?

`CA_snowmelt_april_vs_janmar_match_test.ipynb` found that the Z500/T700 synoptic
heat-wave record matches April rapid-decline events far worse than Feb/Mar ones
(Z500: 38% vs. 88%, p=0.0014), and speculated -- citing Guo and He (2026, *Environ.
Res. Lett.* 21 194013) -- that this might reflect reduced spring snowpack cold
content rather than a weaker atmosphere-snow coupling: by April, an ordinary local
warm spell that never clears the synoptic ridging threshold might already be enough
to trigger rapid melt.

**This notebook tests that directly** by building a Guo-and-He-style *local*
heat-wave definition -- a quantile exceedance of Sierra-box 2m air temperature for a
minimum number of consecutive days (following the general relative-threshold range Guo and He cite from the broader
heatwave-definition literature -- Robinson 2001; Perkins and Alexander 2013: typically
90th/95th percentile, 2-5 consecutive days -- **not a confirmed statement of Guo and
He's own specific method**, which the paper defers to its supporting information S2,
not available here) -- and matching it against the same 41 Jan-Apr rapid-decline melt events,
instead of the synoptic Z500/T700 record.

Self-contained: reads `t2m_sierra_<year>.nc` (`get_t2m_sierra.py`, 1990-2026, same
Sierra box as the snowmelt notebook) and the exported melt-events CSV; detects
threshold choice via `ca_blocking_lib`'s shared climatology/event-detection
machinery, so the only difference from the Z500/T700 comparison is the physical
variable (local T2M vs. synoptic Z500/T700), not the statistical method.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

sys.path.insert(0, str(Path.cwd()))
import ca_blocking_lib as lib

DATA_DIR = Path('/glade/campaign/mmm/c3we/mingge/ERA5/CA_blocking')
YEARS = list(range(1990, 2026))
CLIM_WINDOW = 5
MAX_GAP = 1


## 1. Build the Sierra-box T2M climatology and empirical percentile cutoffs

Detrended calendar-day climatology (same `lib.standardize_with_trend` used for
Z500/T700), fit on 1990-2025. Percentile cutoffs are computed **empirically** from
the fit-year standardized-anomaly distribution rather than assumed from a normal
distribution (90th/95th percentile, following the literature range Guo and He cite -- see intro caveat) --
consistent with this project's established preference for non-parametric checks
(`CA_snowmelt_methodC_5daymean_std.ipynb` Section 10) over assuming normality.

In [2]:
files = [DATA_DIR / f't2m_sierra_{y}.nc' for y in YEARS]
ds = xr.open_mfdataset(files, combine='by_coords', engine='netcdf4')
df = pd.DataFrame({'t2m': ds.t2m.values}, index=pd.to_datetime(ds.time.values)).sort_index()
df = df[df.index.month.isin([1, 2, 3, 4])]

sanom, diag = lib.standardize_with_trend(df.index, df['t2m'].values, window=CLIM_WINDOW, label='t2m')
df['t2m_sanom'] = sanom

P90, P95 = np.percentile(sanom, 90), np.percentile(sanom, 95)
print(f'\nEmpirical sanom percentile cutoffs: 90th={P90:.3f} sigma, 95th={P95:.3f} sigma '
      f'(vs. theoretical normal 1.28/1.645 -- close, T2M anomalies are close to Gaussian)')


ERROR 1: PROJ: proj_create_from_database: Open of /glade/work/mingge/conda-envs/my_npl2026a/share/proj failed


t2m: Theil-Sen trend = +0.213/decade (90% CI [-0.049, +0.454]/decade); OLS = +0.219/decade (p=0.170, r=0.23)
t2m: clim_std, naive (trend-contaminated) = 3.2983, detrended (corrected) = 3.2901 (+0.248% difference)

Empirical sanom percentile cutoffs: 90th=1.264 sigma, 95th=1.517 sigma (vs. theoretical normal 1.28/1.645 -- close, T2M anomalies are close to Gaussian)


## 2. Load the melt events and sweep the percentile x duration grid

The literature range Guo and He cite (90th/95th percentile, 2-5 day minimum persistence,
see intro caveat) -- not a single arbitrary choice. Event-day footprint is reported alongside match rate
throughout, since this project's own permutation-test notebook already showed that
raw match rate alone can't distinguish genuine detection from base-rate inflation.

In [3]:
melt = pd.read_csv(DATA_DIR / 'ca_snotel_snowmelt_events_wy1990-2025.csv',
                   parse_dates=['start_date', 'end_date'])
melt['start_date'] = melt.start_date.dt.date
melt['end_date'] = melt.end_date.dt.date
melt_janapr = melt[melt.start_date.apply(lambda d: d.month in (1, 2, 3, 4))].reset_index(drop=True)
melt_janapr['month'] = melt_janapr.start_date.apply(lambda d: d.month)
melt_janapr['is_april'] = melt_janapr.month == 4
n_total_days = len(df)

grid_rows = []
for p_label, thresh in [('90th pct', P90), ('95th pct', P95)]:
    for min_dur in [2, 3, 5]:
        ev = lib.find_events(df.index, df['t2m_sanom'].values, threshold=thresh,
                              min_duration=min_dur, sign=1, max_gap=MAX_GAP)
        n_event_days = sum((pd.Timestamp(r.end_date) - pd.Timestamp(r.start_date)).days + 1
                            for r in ev.itertuples())
        _, unmatched = lib.overlapping_pairs(melt_janapr, ev, a_prefix='melt', b_prefix='hw')
        n_matched = len(melt_janapr) - len(unmatched)
        grid_rows.append({
            'threshold': p_label, 'min_duration': min_dur, 'n_episodes': len(ev),
            'event_day_pct': 100 * n_event_days / n_total_days,
            'match_rate_pct': 100 * n_matched / len(melt_janapr),
        })

grid_df = pd.DataFrame(grid_rows)
print('For reference: Method A (Z500/T700, >=5-day persistence) event-day footprint is '
      '8%/8% (T700/Z500) with a 61%/59% match rate (CA_snowmelt_events_detrend_5day.ipynb Section 5).')
grid_df

For reference: Method A (Z500/T700, >=5-day persistence) event-day footprint is 8%/8% (T700/Z500) with a 61%/59% match rate (CA_snowmelt_events_detrend_5day.ipynb Section 5).


,threshold,min_duration,n_episodes,event_day_pct,match_rate_pct
0,90th pct,2,101,9.170709,87.804878
1,90th pct,3,66,7.553708,73.170732
2,90th pct,5,25,4.389004,36.585366
3,95th pct,2,54,4.319704,60.975610
4,95th pct,3,31,3.257103,36.585366
5,95th pct,5,11,1.755602,17.073171


## 3. Headline choice: 90th percentile, 2-day minimum persistence

The loosest setting in the literature range Guo and He cite (see intro caveat). Event-day footprint (9%) is nearly
identical to Method A's synoptic footprint (8%) -- so any match-rate improvement here
isn't simply "a denser calendar," the comparison is apples-to-apples on density.

In [4]:
ev_headline = lib.find_events(df.index, df['t2m_sanom'].values, threshold=P90,
                               min_duration=2, sign=1, max_gap=MAX_GAP)
_, unmatched = lib.overlapping_pairs(melt_janapr, ev_headline, a_prefix='melt', b_prefix='hw')
melt_janapr['matched_t2m'] = ~melt_janapr.index.isin(unmatched.index)

rng = np.random.default_rng(42)
lo, hi = lib.boot_ci_proportion(melt_janapr.matched_t2m.values, rng=rng)
print(f'Overall T2M match rate: {melt_janapr.matched_t2m.mean()*100:.0f}% '
      f'(95% bootstrap CI: {lo:.0f}-{hi:.0f}%, n={len(melt_janapr)}) '
      f'-- Method A (Z500/T700) was 61%/59%')

by_month = melt_janapr.groupby('month').agg(n=('matched_t2m', 'size'),
                                             t2m_match_pct=('matched_t2m', lambda s: 100 * s.mean()))
by_month

Overall T2M match rate: 88% (95% bootstrap CI: 76-98%, n=41) -- Method A (Z500/T700) was 61%/59%


,n,t2m_match_pct
month,,
2,6,100.000000
3,11,90.909091
4,24,83.333333


## 4. Formal test: does the April match-rate drop shrink under the local definition?

Same Fisher exact + bootstrap-CI structure as the Z500/T700 version
(`CA_snowmelt_april_vs_janmar_match_test.ipynb` Section 3), for direct comparability.

In [5]:
april = melt_janapr[melt_janapr.is_april]
janmar = melt_janapr[~melt_janapr.is_april]
a_m, a_n = int(april.matched_t2m.sum()), len(april)
j_m, j_n = int(janmar.matched_t2m.sum()), len(janmar)
table = [[a_m, a_n - a_m], [j_m, j_n - j_m]]
odds, p = fisher_exact(table)

a_vals, j_vals = april.matched_t2m.values.astype(bool), janmar.matched_t2m.values.astype(bool)
diffs = np.array([
    rng.choice(a_vals, size=len(a_vals), replace=True).mean()
    - rng.choice(j_vals, size=len(j_vals), replace=True).mean()
    for _ in range(10000)
])
lo_d, hi_d = np.percentile(diffs, [2.5, 97.5])

print(f'T2M (90th pct, 2-day): April {a_m}/{a_n} ({100*a_m/a_n:.0f}%) vs. Jan-Mar {j_m}/{j_n} ({100*j_m/j_n:.0f}%)')
print(f'  Fisher exact: odds ratio={odds:.2f}, p={p:.3f}')
print(f'  Bootstrap 95% CI for (April - Jan-Mar): [{100*lo_d:.0f}, {100*hi_d:.0f}] pts')
print(f'\nFor reference, Z500 (synoptic, >=5-day): April 38% vs. Jan-Mar 88%, p=0.0014, '
      f'CI [-75,-25] pts (CA_snowmelt_april_vs_janmar_match_test.ipynb Section 3)')


T2M (90th pct, 2-day): April 20/24 (83%) vs. Jan-Mar 16/17 (94%)
  Fisher exact: odds ratio=0.31, p=0.382
  Bootstrap 95% CI for (April - Jan-Mar): [-29, 8] pts

For reference, Z500 (synoptic, >=5-day): April 38% vs. Jan-Mar 88%, p=0.0014, CI [-75,-25] pts (CA_snowmelt_april_vs_janmar_match_test.ipynb Section 3)


## 5. Does this hold across the full sensitivity grid, or is it specific to the loosest setting?

Repeats the April-vs-Jan-Mar gap at every percentile x duration combination from
Section 2 -- a single headline number could be cherry-picked, intentionally or not.

In [6]:
grid_gap_rows = []
for p_label, thresh in [('90th pct', P90), ('95th pct', P95)]:
    for min_dur in [2, 3, 5]:
        ev = lib.find_events(df.index, df['t2m_sanom'].values, threshold=thresh,
                              min_duration=min_dur, sign=1, max_gap=MAX_GAP)
        _, unmatched = lib.overlapping_pairs(melt_janapr, ev, a_prefix='melt', b_prefix='hw')
        mj = melt_janapr.copy()
        mj['matched'] = ~mj.index.isin(unmatched.index)
        apr_rate = 100 * mj[mj.is_april].matched.mean()
        jm_rate = 100 * mj[~mj.is_april].matched.mean()
        grid_gap_rows.append({'threshold': p_label, 'min_duration': min_dur,
                               'april_pct': apr_rate, 'janmar_pct': jm_rate,
                               'gap_pts': jm_rate - apr_rate})

grid_gap_df = pd.DataFrame(grid_gap_rows)
print('April-vs-Jan-Mar gap across the full sensitivity grid (Z500 synoptic gap was 50 pts, '
      'T700 was 26 pts, for reference):')
grid_gap_df

April-vs-Jan-Mar gap across the full sensitivity grid (Z500 synoptic gap was 50 pts, T700 was 26 pts, for reference):


,threshold,min_duration,april_pct,janmar_pct,gap_pts
0,90th pct,2,83.333333,94.117647,10.784314
1,90th pct,3,66.666667,82.352941,15.686275
2,90th pct,5,20.833333,58.823529,37.990196
3,95th pct,2,50.000000,76.470588,26.470588
4,95th pct,3,29.166667,47.058824,17.892157
5,95th pct,5,8.333333,29.411765,21.078431


## 6. Conclusion

**The headline result confirms the Guo and He hypothesis, but the sensitivity grid
reveals it's more specific than "local warmth explains April better" -- it's
specifically *short* local warm pulses that do.**

| Definition | overall match | April | Jan-Mar | gap | April-vs-JanMar p |
|---|---|---|---|---|---|
| T2M, 90th pct, 2-day (headline) | 88% [76,98] | 83% | 94% | 11 pts | 0.38 (not sig.) |
| Z500, synoptic, >=5-day | 59% | 38% | 88% | 50 pts | **0.0014** |
| T700, synoptic, >=5-day | 61% | 50% | 76% | 26 pts | 0.11 |

- **At the loosest setting (90th percentile, 2-day minimum -- the low end of the literature range Guo and
  He cite, not confirmed to be their own specific choice), the April problem essentially disappears.** Overall match
  rate (88%) is far higher than either synoptic variable's (59-61%), with comparable
  event-day footprint (9% vs. Method A's 8% -- not a denser-calendar artifact), and
  the April-vs-Jan-Mar gap (11 pts) is not statistically distinguishable from zero.
- **But the gap re-opens, and grows substantially, as soon as persistence is
  required.** Walking the same definition from 2-day to 5-day minimum persistence
  (Section 5): the April gap grows from 11 to 16 to **38 points** at 90th percentile,
  and from 26 to 18 to 21 points at 95th percentile. At 5-day minimum -- the same
  persistence Z500/T700 require -- the local-temperature definition's April gap (21-38
  pts) is in the same range as the synoptic definitions' (26-50 pts), not meaningfully
  smaller.
- **This refines, rather than simply confirms, the cold-content hypothesis.** The
  finding isn't "local warmth matters more than synoptic ridging in general" -- it's
  that April rapid-decline events are disproportionately triggered by *brief* (1-2
  day) local warm pulses that never need to persist, consistent with Guo and He's
  mechanism (reduced spring cold content lowers the energy needed to trigger melt, so
  a short warm spell is already sufficient). Once *persistence* is required -- whether
  defined locally or synoptically -- April stops looking special, because a 5-day
  sustained top-10%-warm spell is itself a fairly rare, quasi-synoptic event (local
  temperature is far more day-to-day volatile than a 500 hPa height field, so
  "5 straight warm days" at the surface increasingly implies an underlying persistent
  warm regime rather than a short radiative/advective pulse) -- the two timescales
  converge toward detecting the same underlying phenomenon.
- **Practical implication for the project's framing.** The April match-rate drop
  found in the Z500/T700 analysis is not evidence that the atmosphere-snow coupling
  breaks down in April, and not fully explained by "local warmth suffices" either --
  it's specifically a *persistence-threshold* artifact: April melt doesn't need a
  multi-day heat wave the way Feb/Mar melt apparently does, because a day or two of
  ordinary warmth is already enough once the snowpack is primed. A detector built
  around multi-day persistence (whether Z500/T700's 5-day rule or this notebook's own
  3-5 day local-T2M sweep) will structurally under-detect this shorter-fuse spring
  mechanism, independent of which variable it's applied to.

**Caveat.** This uses a single Sierra-box-averaged T2M series, not Guo and He's actual
per-station design (823 individual SNOTEL gridcells) -- a reasonable simplification
given this project's existing convention of box-averaged indices throughout, but it
means station-to-station heterogeneity in the local-warmth/melt relationship (e.g. by
elevation or aspect) isn't tested here.